# TK 1 Analisis Numerik — Nomor 1
## Analisis Perpindahan Penumpang Antarhalte

## 0. Setup dan Konfigurasi

Jalankan notebook dari direktori proyek; dataset berada di `data/T_<N>.csv`.
`TOL` dapat dikonfigurasi sesuai kebutuhan toleransi numerik. Vektor menggunakan bentuk `(N,)`.

In [1]:
from pathlib import Path
import numpy as np

DATA_DIR = Path("data")
MATRIX_SIZES = [16, 32, 64, 128, 256, 512]
TOL = 1e-10


def max_abs_diff(x, y):
    """Selisih absolut maksimum dua vektor berbentuk (N,)."""
    return float(np.max(np.abs(x - y)))


**Output/check:** Sel setup mendefinisikan konfigurasi dan helper; tidak menghasilkan output.

## 1. Validasi Data

Setiap `data/T_<N>.csv` dibaca tanpa header. Matriks harus berdimensi
$N\times N$, memiliki elemen berhingga dan $T_{ij}\geq 0$, serta memenuhi
$\left|\sum_j T_{ij}-1\right|\leq\mathrm{TOL}$ untuk setiap baris.
Nonnegativitas diperiksa secara ketat; data tidak dipotong atau dinormalisasi ulang.

Untuk rantai Markov berhingga, distribusi stasioner tunggal **jika dan hanya jika**
terdapat tepat satu kelas komunikasi tertutup. Kelas komunikasi diperoleh dari
komponen terhubung kuat (SCC) graf berarah dengan sisi $i\to j$ jika $T_{ij}>0$.
SCC tertutup tidak memiliki sisi menuju SCC lain. Graf yang terhubung kuat
(irreducible) merupakan syarat cukup, tetapi bukan syarat perlu untuk ketunggalan.

Aperiodisitas diperlukan untuk konvergensi menuju distribusi stasioner dari setiap
kondisi awal, bukan untuk ketunggalan saja. Pada kelas tertutup yang irreducible,
satu self-loop positif sudah menjamin periode satu. Pemeriksaan ini merupakan
syarat cukup; jika tidak ada self-loop, periode belum disimpulkan.
Probabilitas positif sekecil apa pun tetap menjadi sisi; `TOL` hanya dipakai untuk
jumlah baris. Tidak ada solver atau faktorisasi yang digunakan.


In [2]:
def communication_classes(T):
    """SCC graf transisi dengan dua penelusuran DFS (Kosaraju)."""
    n = T.shape[0]
    outgoing = [np.flatnonzero(T[i] > 0).tolist() for i in range(n)]
    incoming = [[] for _ in range(n)]
    for i, neighbors in enumerate(outgoing):
        for j in neighbors:
            incoming[j].append(i)

    seen, order = set(), []
    for root in range(n):
        if root in seen:
            continue
        seen.add(root)
        stack = [(root, iter(outgoing[root]))]
        while stack:
            vertex, neighbors = stack[-1]
            neighbor = next(neighbors, None)
            if neighbor is None:
                order.append(vertex)
                stack.pop()
            elif neighbor not in seen:
                seen.add(neighbor)
                stack.append((neighbor, iter(outgoing[neighbor])))

    labels = [-1] * n
    classes = []
    for root in reversed(order):
        if labels[root] != -1:
            continue
        label = len(classes)
        labels[root] = label
        stack, members = [root], []
        while stack:
            vertex = stack.pop()
            members.append(vertex)
            for neighbor in incoming[vertex]:
                if labels[neighbor] == -1:
                    labels[neighbor] = label
                    stack.append(neighbor)
        classes.append(members)

    closed = [members for label, members in enumerate(classes)
              if all(labels[j] == label for i in members for j in outgoing[i])]
    return classes, closed


def validate_transition_matrix(T, N, tol=TOL):
    """Validasi T tanpa mengubah data; tolak matriks transisi yang tidak sah."""
    if not np.isfinite(tol) or tol < 0:
        raise ValueError("Toleransi harus berhingga dan nonnegatif.")
    if T.shape != (N, N) or N < 1:
        raise ValueError(f"Dimensi {T.shape}, seharusnya ({N}, {N}).")
    if not np.all(np.isfinite(T)):
        raise ValueError("Matriks mengandung NaN atau infinity.")
    minimum = float(np.min(T))
    if minimum < 0:
        raise ValueError(f"Elemen negatif ditemukan: minimum = {minimum}.")
    row_sums = np.sum(T, axis=1)
    row_error = float(np.max(np.abs(row_sums - 1.0)))
    if row_error > tol:
        raise ValueError(f"Galat jumlah baris {row_error:.3e} melebihi {tol:.3e}.")
    classes, closed = communication_classes(T)
    return {
        "shape": T.shape,
        "minimum": minimum,
        "row_sums": row_sums,
        "max_row_sum_error": row_error,
        "communication_classes": classes,
        "closed_classes": closed,
        "irreducible": len(classes) == 1,
        "unique_stationary": len(closed) == 1,
        "aperiodic_closed_classes_confirmed": all(
            any(T[i, i] > 0 for i in members) for members in closed
        ),
    }


def load_transition_matrix(path, N, tol=TOL):
    """Baca CSV numerik dan kembalikan (T, hasil_validasi)."""
    T = np.loadtxt(path, delimiter=",", ndmin=2)
    return T, validate_transition_matrix(T, N, tol)


matrices = {}
validation_results = {}
print("N    dimensi      min(T)     galat baris   SCC  tertutup  tunggal  aperiodik*")
for N in MATRIX_SIZES:
    path = DATA_DIR / f"T_{N}.csv"
    T, result = load_transition_matrix(path, N)
    matrices[N] = T
    validation_results[N] = result
    print(f"{N:<4} {str(T.shape):<12} {result['minimum']:.1e}    "
          f"{result['max_row_sum_error']:.3e}     "
          f"{len(result['communication_classes']):<4} "
          f"{len(result['closed_classes']):<9} "
          f"{str(result['unique_stationary']):<8} "
          f"{result['aperiodic_closed_classes_confirmed']}")
print("* True: self-loop membuktikan aperiodisitas kelas tertutup; False: belum disimpulkan.")


N    dimensi      min(T)     galat baris   SCC  tertutup  tunggal  aperiodik*
16   (16, 16)     0.0e+00    1.110e-16     1    1         True     True
32   (32, 32)     0.0e+00    2.220e-16     1    1         True     True
64   (64, 64)     0.0e+00    1.110e-16     1    1         True     True
128  (128, 128)   0.0e+00    2.220e-16     1    1         True     True
256  (256, 256)   0.0e+00    2.220e-16     1    1         True     True
512  (512, 512)   0.0e+00    2.220e-16     1    1         True     True
* True: self-loop membuktikan aperiodisitas kelas tertutup; False: belum disimpulkan.


**Output/check:** Tabel di atas melaporkan semua ukuran dataset. `matrices[N]`
menyimpan matriks asli dan `validation_results[N]` menyimpan diagnostik, termasuk
vektor jumlah baris berbentuk `(N,)`. Sel berikut menguji graf irreducible,
graf reducible dengan satu/dua kelas tertutup, rantai periodik, serta data tidak sah.

In [3]:
def check_data_validation():
    examples = [
        (np.array([[0.5, 0.5], [0.5, 0.5]]), True, True, True),
        (np.array([[0.0, 1.0], [0.0, 1.0]]), False, True, True),
        (np.array([[1.0, 0.0], [0.0, 1.0]]), False, False, True),
        (np.array([[0.0, 1.0], [1.0, 0.0]]), True, True, False),
        (np.array([[1.0 - 1e-12, 1e-12], [1e-12, 1.0 - 1e-12]]),
         True, True, True),
    ]
    for T, irreducible, unique, aperiodic in examples:
        result = validate_transition_matrix(T, 2)
        assert result["irreducible"] == irreducible
        assert result["unique_stationary"] == unique
        assert result["aperiodic_closed_classes_confirmed"] == aperiodic
        assert result["row_sums"].shape == (2,)
    for T in [np.zeros((2, 3)), np.array([[float("nan"), 0], [0, 1]]),
              np.array([[float("inf"), 0], [0, 1]]),
              np.array([[1.1, -0.1], [0, 1]]),
              np.array([[0.4, 0.4], [0, 1]])]:
        try:
            validate_transition_matrix(T, 2)
        except ValueError:
            pass
        else:
            raise AssertionError("Matriks tidak sah seharusnya ditolak.")
    print("Self-check validasi data: OK")


check_data_validation()


Self-check validasi data: OK


## 2. Formulasi dan Penanganan Singularitas

Penjelasan akan diisi pada fase berikutnya.

**Output/check:** Akan diisi pada fase berikutnya.

## 3. Identifikasi Struktur Matriks

Penjelasan akan diisi pada fase berikutnya.

**Output/check:** Akan diisi pada fase berikutnya.

## 4. Dense LU + Partial Pivoting

Penjelasan akan diisi pada fase berikutnya.

**Output/check:** Akan diisi pada fase berikutnya.

## 5. Banded/Tridiagonal Solver + Partial Pivoting

Penjelasan akan diisi pada fase berikutnya.

**Output/check:** Akan diisi pada fase berikutnya.

## 6. Verification / Handoff Check

Penjelasan akan diisi pada fase berikutnya.

**Output/check:** Akan diisi pada fase berikutnya.

## 7. Handoff Notes

Penjelasan akan diisi pada fase berikutnya.

**Output/check:** Akan diisi pada fase berikutnya.